In [1]:
import sys
from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation_functional as sim
import xobjects as xo
import copy

In [6]:
gemitt = 60e-6
n_turns=100
n_particles = 1000

line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

s_list = np.linspace(0, line.get_length(), 26, endpoint=False)
strength_list = 1e-3 * np.cos(13 * 2*np.pi * s_list / line.get_length())
line = sim.add_dipole(line, strength=strength_list, s=s_list, mode='skew')

tw = line.twiss(method='4d', freeze_longitudinal=True)
bg = line.particle_ref.beta0*line.particle_ref.gamma0
particles_base = line.build_particles(
        x_norm=np.random.normal(size=n_particles),
        y_norm=np.random.normal(size=n_particles),
        px_norm=np.random.normal(size=n_particles),
        py_norm=np.random.normal(size=n_particles),
        nemitt_x=bg*gemitt,
        nemitt_y=bg*gemitt,
        method='4d',
        freeze_longitudinal=True,
        mode='normalized_transverse')
particles_base.x += (tw.x[0] - np.mean(particles_base.x))
particles_base.y += (tw.y[0] - np.mean(particles_base.y))
particles_base.px += (tw.px[0] - np.mean(particles_base.px))
particles_base.py += (tw.py[0] - np.mean(particles_base.py))

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [10]:
for angle in ['normal', 'skew']:
    for ind in [False, True]:
        line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))
        
        s_list = np.linspace(0, line.get_length(), 26, endpoint=False)
        strength_list = 5e-3 * np.cos(13 * 2*np.pi * s_list / line.get_length())
        line = sim.add_dipole(line, strength=strength_list, s=s_list, mode=angle)
        
        particles = copy.deepcopy(particles_base)
        
        line = sim.add_monitors(line, particles, n_monitors=100, n_turns=n_turns)
        
        twiss = line.twiss(method='4d', freeze_longitudinal=True)
        
        line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=1e13,
                                                     n_interactions=100, 
                                                     gemitt=gemitt, indirect=ind)
        
        line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
        line.track(particles, num_turns=n_turns, time=True)
        
        sim.save(line, particles, twiss, filename=f'data/{angle}_ind_{ind}', aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/4126 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4716 [00:00<?, ?it/s]

...

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/4126 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4716 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/skew_ind_True
